# Methods for Running Experiments

There are four main ways to run experiments in this framework. Each has different advantages depending on your needs.

## Overview of Methods

| Method | Best For | Complexity | Flexibility |
|--------|----------|------------|-------------|
| **1. CLI Tool** | Quick execution, exploring | ⭐ Easy | ⭐ Low |
| **2. Direct Execution** | Simple one-off runs | ⭐⭐ Easy | ⭐⭐ Medium |
| **3. Custom Scripts** | Parameter variations, batch jobs | ⭐⭐⭐ Medium | ⭐⭐⭐⭐ High |
| **4. Interactive Python** | Exploration, debugging | ⭐⭐ Easy | ⭐⭐⭐⭐⭐ Very High |

## Method 1: CLI Tool (Recommended for Beginners)

The command-line interface provides the easiest way to run pre-built experiments.

### Available Commands

```bash
# List all available experiments
python run_experiment.py list

# Get info about a specific experiment
python run_experiment.py info mock_experiment

# Run an experiment
python run_experiment.py mock_experiment
```

### Examples

In [1]:
# Simulate running the CLI command (from Python)
import subprocess
import sys
from pathlib import Path

# Navigate to parent directory if needed
parent_dir = Path.cwd().parent if 'jupyter_book_tutorial' in str(Path.cwd()) else Path.cwd()

# List available experiments
result = subprocess.run(
    [sys.executable, 'run_experiment.py', 'list'],
    cwd=parent_dir,
    capture_output=True,
    text=True
)

print(result.stdout)


AVAILABLE EXPERIMENTS

  adaptive_tuning
    Adaptive Tuning Experiment

  fm_band_survey
    FM Band Survey Experiment

  fm_quick_test
    FM SHM Quick Test

  fm_shm_performance_test
    FM SHM Performance Test Experiment

  fm_station_monitor
    FM Station Monitoring Experiment

  frequency_characterization
    Frequency Characterization Experiment

  grid_scan_experiment
    Grid Scan Experiment (2D Parameter Space)

  mock_experiment
    Mock Experiment (No Hardware Required)

  ottawa_fm_comprehensive_test
    Comprehensive Ottawa FM Station Test

  peak_finding_experiment
    Peak Finding Experiment

  signal_quality_scan
    Signal Quality Scan Experiment

  synchronized_sweep
    Synchronized TX/RX Sweep Experiment


Total: 12 experiments

Run an experiment with:
  python run_experiment.py <experiment_name>

Example:
  python run_experiment.py mock_experiment




### Pros and Cons

**Advantages:**
- ✅ Simple and fast
- ✅ Auto-discovery of experiments
- ✅ Built-in error handling
- ✅ No code editing needed

**Disadvantages:**
- ❌ Limited customization
- ❌ Can't modify parameters without editing experiment file
- ❌ Less flexible than other methods

## Method 2: Direct Python Execution

Run experiment scripts directly with Python.

### Example

```bash
cd experiments
python mock_experiment.py
```

Let's simulate this:

In [2]:
# Run an experiment script directly
result = subprocess.run(
    [sys.executable, 'experiments/mock_experiment.py'],
    cwd=parent_dir,
    capture_output=True,
    text=True,
    timeout=30
)

print("Output:")
print(result.stdout)
if result.stderr:
    print("\nErrors/Warnings:")
    print(result.stderr)

Output:

MOCK EXPERIMENT (No Hardware Required)
Experiment ID: MOCK-001
Operator: yak
Purpose: Test Bluesky framework with mock devices

Motor range: 0 - 10
Number of points: 20



Transient Scan ID: 1     Time: 2025-10-19 11:49:00
Persistent Unique Scan ID: '880961e7-713e-4950-a650-478ef73e9ca0'

🔍 Peak Detector: Monitoring 'det_value'
   Alert threshold: 1.2

📈 Computing statistics for: det_value

⏱️  Scan started: scan
📝 Logging documents to: 880961e7_documents.jsonl
New stream: 'primary'
+-----------+------------+------------+------------+
|   seq_num |       time |      motor |  det_value |
+-----------+------------+------------+------------+
   Position signal: 'motor' (auto-detected)
|         1 | 11:49:00.6 |      0.000 |      0.871 |

   Events: 1|         2 | 11:49:00.6 |      0.526 |      0.787 |

   Events: 2|         3 | 11:49:00.6 |      1.053 |      1.340 |
   🔔 ALERT: det_value = 1.3401 exceeds threshold 1.2 at motor = 1.0526e+00

   Events: 3|         4 | 11:49:00.6 | 

### Pros and Cons

**Advantages:**
- ✅ Simple
- ✅ Direct access to experiment file
- ✅ Can edit file for quick parameter changes

**Disadvantages:**
- ❌ Must be in correct directory
- ❌ Requires file editing for different parameters
- ❌ Less structured than CLI

## Method 3: Custom Python Scripts

Create your own scripts with full control over parameters, callbacks, and workflow.

### Example: Custom Parameter Scan

In [3]:
# Setup imports (add parent to path if needed)
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from bluesky import RunEngine
from ophyd import Signal
from bluesky.plans import scan
from bluesky_config.devices import MockDetector
from bluesky_config.callbacks import DocumentLogger, StatisticsCallback
from bluesky_config.metadata import create_experiment_metadata

# Setup
RE = RunEngine({})
doc_logger = DocumentLogger(str(parent_dir / 'data' / 'documents'))
RE.subscribe(doc_logger)

stats = StatisticsCallback(['det_value'])
RE.subscribe(stats)

# Create devices
motor = Signal(name='motor', value=0)
detector = MockDetector(name='det', noise_level=0.1)

# CUSTOMIZE PARAMETERS HERE!
start = 0
stop = 20  # Different from default
num_points = 50  # More points

# Create metadata
md = create_experiment_metadata(
    experiment_id='CUSTOM-001',
    purpose='Custom parameter scan with 50 points',
    plan_type='scan',
    start=start,
    stop=stop,
    num_points=num_points,
)

# Run the scan
print(f"Running custom scan: {start} to {stop} with {num_points} points")
uid = RE(scan([detector], motor, start, stop, num_points), **md)

print(f"\n✓ Complete! Run UID: {uid[0][:8]}")
print(f"\nStatistics:")
print(f"  Mean: {stats.mean['det_value']:.3f}")
print(f"  Std Dev: {stats.std['det_value']:.3f}")

Running custom scan: 0 to 20 with 50 points

📝 Logging documents to: 7fe914bb_documents.jsonl


📈 Computing statistics for: det_value

   Saved stop document


   📊 Statistics Summary:

...

         Max:    1.2093

         Median: 0.9783


✓ Complete! Run UID: 7fe914bb


Statistics:

AttributeError: 'StatisticsCallback' object has no attribute 'mean'

### Batch Processing Example

Custom scripts are perfect for running multiple experiments with parameter variations:

In [ ]:
# Batch processing: Run 3 scans with different noise levels
noise_levels = [0.1, 0.3, 0.5]
uids = []

for i, noise in enumerate(noise_levels, 1):
    # Update detector noise
    detector.noise_level = noise
    
    # Create metadata
    md = create_experiment_metadata(
        experiment_id=f'BATCH-{i:03d}',
        purpose=f'Batch scan {i} with noise={noise}',
        plan_type='scan',
        batch_id=i,
        noise_level=noise,
    )
    
    # Run scan
    uid = RE(scan([detector], motor, 0, 10, 20), **md)
    uids.append(uid[0])
    print(f"Run {i}/3 complete: {uid[0][:8]} (noise={noise})")

print(f"\n✓ All 3 batch runs complete!")
print(f"UIDs: {[uid[:8] for uid in uids]}")

### Pros and Cons

**Advantages:**
- ✅ Full control over parameters
- ✅ Can customize callbacks and metadata
- ✅ Easy to modify and iterate
- ✅ Can run multiple scans in sequence
- ✅ Perfect for batch processing

**Disadvantages:**
- ❌ Requires Python knowledge
- ❌ More verbose than CLI
- ❌ Need to manage imports and setup

## Method 4: Interactive Python Session

Use an interactive Python session (or this notebook!) for exploration and debugging.

### Example: Interactive Exploration

In [ ]:
# Interactive approach - try different things step by step
from databroker import Broker

# Get DataBroker
db = Broker.named('temp')

# Run a quick scan
motor.set(0)  # Reset position
uid = RE(scan([detector], motor, 0, 5, 10))

# IMMEDIATELY retrieve and inspect
run = db[-1]
table = run.table()

print("Quick inspection:")
print(f"Shape: {table.shape}")
print(f"\nFirst few rows:")
print(table.head())

In [ ]:
# Try a different scan with different parameters
detector.noise_level = 0.05  # Less noise
uid2 = RE(scan([detector], motor, 0, 10, 30))  # More points

# Compare the two
run2 = db[-1]
table2 = run2.table()

print("Comparison:")
print(f"Run 1: {table.shape[0]} points, mean={table['det_value'].mean():.3f}")
print(f"Run 2: {table2.shape[0]} points, mean={table2['det_value'].mean():.3f}")

In [ ]:
# Quick plot comparison
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(table['motor'], table['det_value'], 'o-', label='Run 1 (high noise)')
ax1.set_xlabel('Motor')
ax1.set_ylabel('Detector Value')
ax1.set_title('Run 1: 10 points')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(table2['motor'], table2['det_value'], 'o-', color='orange', label='Run 2 (low noise)')
ax2.set_xlabel('Motor')
ax2.set_ylabel('Detector Value')
ax2.set_title('Run 2: 30 points')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

### Pros and Cons

**Advantages:**
- ✅ Immediate feedback
- ✅ Can inspect variables at any point
- ✅ Great for learning and debugging
- ✅ Quick iterations
- ✅ Perfect for this Jupyter Book!

**Disadvantages:**
- ❌ Not reproducible (unless you save commands)
- ❌ Easy to lose track of what you did
- ❌ No persistent script

## Comparison Summary

### When to Use Each Method

**Use CLI Tool when:**
- You're just getting started
- You want to quickly run a pre-built experiment
- You don't need to customize parameters

**Use Direct Execution when:**
- You have a specific experiment script
- You only need minor parameter tweaks
- You're working from the terminal

**Use Custom Scripts when:**
- You need custom parameter combinations
- You're running batch jobs
- You want to save your workflow
- You need specific callback configurations

**Use Interactive Python when:**
- You're exploring and learning
- You're debugging issues
- You want immediate feedback
- You're in a Jupyter notebook (like now!)

## Essential Pattern (All Methods)

No matter which method you use, data capture always follows this pattern:

```python
# 1. Create RunEngine
RE = RunEngine({})

# 2. Subscribe callbacks (CRITICAL for data capture!)
doc_logger = DocumentLogger('data/documents')
RE.subscribe(doc_logger)

# 3. Create devices
motor = Signal(name='motor', value=0)
detector = MockDetector(name='det')

# 4. Run experiment
uid = RE(scan([detector], motor, 0, 10, 20))

# 5. Data is automatically saved!
```

```{important}
The **DocumentLogger subscription** is what enables data capture. 
Without it, your data won't be saved to disk!
```

## Try It Yourself

**Exercise 1**: Modify the custom script above to scan from 0 to 30 with 100 points.

**Exercise 2**: Run 5 batch scans with noise levels: [0.05, 0.1, 0.2, 0.4, 0.8]

**Exercise 3**: In interactive mode, run scans with different numbers of points and compare the results.

## Next Steps

Now that you know how to run experiments, let's learn about what data actually gets captured:

→ **Chapter 3: [What Gets Captured](03_data_capture.ipynb)**